# conDitar-dev · CSF1R 소규모 Colab 데모 · v001

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HisameOgasahara/tmp_protein/blob/main/conDitar_CSF1R_Colab.ipynb)

CSF1R 결정구조 8W1L → 사전학습 diffusion으로 새 분자 생성 → 분자 구조·물성 및 Vina 점수 확인 → 논문 CSF1R 억제 실험과 비교 → ZIP 다운로드.

Colab에서 **런타임 → 런타임 유형 변경 → T4 GPU**를 선택합니다. 기본값은 분자 5개, 배치 1개입니다. 저자의 diffusion step과 seed를 유지합니다. T4 실측은 아래 셀에 기록됩니다.

[공식 코드 및 사용 조건](https://github.com/ninglab/conDitar-dev/blob/main/scripts/conDitar/LICENCE.txt) · [논문 Figure 8 및 Table 6](https://arxiv.org/html/2607.12349v1#S5) · [8W1L 결정구조](https://www.rcsb.org/structure/8W1L)

공식 코드 및 가중치는 실행할 때 각 출처에서 받습니다. 논문 실험값은 논문의 후보 ID에만 연결합니다.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import os
import platform
import shutil
import subprocess
import time

if not Path('/content').is_dir() or not shutil.which('nvidia-smi'):
    raise RuntimeError('Google Colab GPU 런타임에서 실행하세요.')

BASE = Path('/content/conditar_v001')
RUN = BASE / datetime.now(timezone.utc).strftime('run_%Y%m%d_%H%M%S_%f')
OUTPUT = RUN / 'results'
INPUTS = OUTPUT / 'inputs'
SAMPLE_ROOT = OUTPUT / 'generated'
SAMPLES = SAMPLE_ROOT / '8w1l'
RUNTIME = OUTPUT / 'runtime'
for directory in (INPUTS, SAMPLES, RUNTIME):
    directory.mkdir(parents=True)

REPO = BASE / 'upstream'
REVISION = '8b322d0f08b973bce4c8e30d8517e30b5c65f342'
NUM_SAMPLES = 5
BATCH_SIZE = 1
ENV_DIR = BASE / 'env'
PYTHON = ENV_DIR / 'bin/python'
MAMBA = BASE / 'bin/micromamba'
ENV = dict(os.environ, MAMBA_ROOT_PREFIX=str(BASE / 'mamba'), PYTHONUNBUFFERED='1')

def run(args, **kwargs):
    subprocess.run([str(arg) for arg in args], env=ENV, check=True, **kwargs)

def download(url, path):
    partial = path.with_name(path.name + '.part')
    run(['curl', '--fail', '--location', '--retry', '3', '--silent', '--show-error',
         '--output', partial, url])
    partial.replace(path)

gpu = subprocess.check_output(['nvidia-smi', '--query-gpu=name,driver_version,memory.total',
                               '--format=csv'], text=True)
print(gpu)
(RUNTIME / 'host.txt').write_text(platform.python_version() + '\n' + gpu)
if not (REPO / '.git').is_dir():
    run(['git', 'clone', 'https://github.com/ninglab/conDitar-dev.git', REPO])
run(['git', '-C', REPO, 'checkout', '--detach', REVISION])

if not MAMBA.exists():
    archive = BASE / 'micromamba.tar.bz2'
    download('https://micro.mamba.pm/api/micromamba/linux-64/latest', archive)
    run(['tar', '-xjf', archive, '-C', BASE, 'bin/micromamba'])
if not PYTHON.exists():
    run([MAMBA, 'create', '-y', '-p', ENV_DIR, '-f', REPO / 'docker/environment.yml'])
ENV['PATH'] = str(PYTHON.parent) + os.pathsep + ENV['PATH']
run([PYTHON, '-m', 'pip', 'install', 'uv'])
uv = [PYTHON, '-m', 'uv', 'pip', 'install', '--python', PYTHON]
run([*uv, 'torch==2.5.1', '--index-url', 'https://download.pytorch.org/whl/cu124'])
run([*uv, 'torch-geometric==2.7.0', 'torch-scatter==2.1.2', 'torch-sparse==0.6.18',
     'torch-cluster==1.6.3', 'torch-spline-conv==1.2.2', '--only-binary', ':all:',
     '--find-links', 'https://data.pyg.org/whl/torch-2.5.1+cu124.html'])
run([*uv, 'meeko==0.1.dev3', 'pdb2pqr', 'tqdm', 'vina', 'cvxpy', 'biopython',
     'git+https://github.com/Valdes-Tresanco-MS/AutoDockTools_py3'])
run([PYTHON, '-c', 'import torch, torch_cluster, torch_scatter; assert torch.cuda.is_available(); print(torch.__version__, torch.version.cuda, torch.cuda.get_device_name()); from scripts.conDitar import sample'], cwd=REPO)
(RUNTIME / 'packages.txt').write_text(subprocess.check_output(
    [str(PYTHON), '-m', 'pip', 'freeze'], text=True, env=ENV))


In [ ]:
from hashlib import file_digest

CHECKPOINTS = BASE / 'checkpoints'
CHECKPOINTS.mkdir(exist_ok=True)
weights = {
    'Diff.pt': ('1Jv4NDVmyRoH_-CmCuWJe9ZIdS6XzPwQ9', 125799286),
    'PocketAE.pt': ('1c046SsBpVz96ugJyiRyaXyEtxJEc5n_E', 39516702),
}
weight_records = {}
for name, (file_id, expected_bytes) in weights.items():
    path = CHECKPOINTS / name
    if not path.exists() or path.stat().st_size != expected_bytes:
        download(f'https://drive.usercontent.google.com/download?id={file_id}&export=download&confirm=t', path)
    if path.stat().st_size != expected_bytes:
        raise RuntimeError(f'{name} 다운로드 크기 불일치')
    with path.open('rb') as stream:
        weight_records[name] = {'drive_id': file_id, 'bytes': expected_bytes,
                                'sha256': file_digest(stream, 'sha256').hexdigest()}
    print(name, f'{expected_bytes / 1e6:.1f} MB')

TARGET = INPUTS / '8w1l'
TARGET.mkdir(exist_ok=True)
download('https://files.rcsb.org/download/8W1L.cif', TARGET / '8w1l.cif')
download('https://models.rcsb.org/v1/8w1l/ligand?auth_asym_id=A&auth_seq_id=1001&encoding=sdf',
         TARGET / '8w1l.sdf')
CONFIG = OUTPUT / 'sample.yml'
run([PYTHON, '-c', '''
from pathlib import Path
import sys
import yaml
from Bio.PDB import MMCIFParser, PDBIO, Select
from rdkit import Chem

source, config_path, checkpoint_dir, target = map(Path, sys.argv[1:])
structure = MMCIFParser(QUIET=True).get_structure("8w1l", target / "8w1l.cif")

class ProteinChainA(Select):
    def accept_chain(self, chain):
        return chain.id == "A"

    def accept_residue(self, residue):
        return residue.id[0] == " "

writer = PDBIO()
writer.set_structure(structure[0])
writer.save(str(target / "8w1l.pdb"), ProteinChainA())
assert any(line.startswith("ATOM") for line in (target / "8w1l.pdb").read_text().splitlines())
assert Chem.SDMolSupplier(str(target / "8w1l.sdf"))[0] is not None
config = yaml.safe_load(source.read_text())
config["model"]["checkpoint"] = str(checkpoint_dir / "Diff.pt")
config["model"]["checkpoint_pocket"] = str(checkpoint_dir / "PocketAE.pt")
config_path.write_text(yaml.safe_dump(config, sort_keys=False))
''', REPO / 'configs/sample_container.yml', CONFIG, CHECKPOINTS, TARGET])
(RUNTIME / 'settings.json').write_text(json.dumps({
    'upstream_commit': REVISION, 'weights': weight_records, 'pdb': '8W1L',
    'reference_ligand': 'A1AE3', 'num_samples': NUM_SAMPLES, 'batch_size': BATCH_SIZE,
    'input_preparation': 'RCSB mmCIF chain A standard residues and bound reference SDF',
}, indent=2))


### T4에서 3D 후보 생성

포켓 인코더가 8W1L의 결합 부위를 읽고, diffusion 모델이 새 분자의 원자와 3D 좌표를 생성합니다. 생성 시도 중 분자로 복원된 결과가 SDF에 저장됩니다. GPU 사용량은 1초 간격으로 샘플링합니다.

In [ ]:
def stage(name, args):
    start = time.perf_counter()
    monitor_path = RUNTIME / f'{name}_gpu.csv'
    with monitor_path.open('w') as gpu_log, (RUNTIME / f'{name}.log').open('w') as log:
        monitor = subprocess.Popen(
            ['nvidia-smi', '--query-gpu=timestamp,memory.used', '--format=csv,noheader,nounits', '-l', '1'],
            stdout=gpu_log)
        try:
            process = subprocess.Popen([str(arg) for arg in args], cwd=REPO, env=ENV,
                                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                       text=True, errors='replace')
            for line in process.stdout:
                print(line, end='')
                log.write(line)
            status = process.wait()
        finally:
            monitor.terminate()
            monitor.wait()
    measurements = [int(line.rsplit(',', 1)[-1]) for line in monitor_path.read_text().splitlines()
                    if line.rsplit(',', 1)[-1].strip().isdigit()]
    stats = {'seconds': round(time.perf_counter() - start, 2),
             'device_peak_sampled_mib': max(measurements, default=None), 'exit_code': status}
    (RUNTIME / f'{name}.json').write_text(json.dumps(stats, indent=2))
    print(stats)
    if status:
        raise subprocess.CalledProcessError(status, args)

stage('generation', [PYTHON, '-m', 'scripts.conDitar.sample', CONFIG,
      '--device', 'cuda:0', '--protein_root', INPUTS, '--pdb_filename', '8w1l/8w1l.pdb',
      '--sdf_filename', '8w1l/8w1l.sdf', '--pocket_radius', '10',
      '--num_samples', NUM_SAMPLES, '--batch_size', BATCH_SIZE, '--result_path', SAMPLES,
      '--tmp_dir', RUN / 'sample_tmp'])
generated = sorted(SAMPLES.glob('*_generated_*.sdf'))
if not generated:
    raise RuntimeError('복원된 분자가 없습니다. generation.log를 확인하세요.')
print('저장된 분자:', len(generated))


In [ ]:
from IPython.display import Image, display
import pandas as pd

run([PYTHON, '-c', '''
from pathlib import Path
import csv
import sys
from rdkit import Chem
from rdkit.Chem import Descriptors, Draw, QED

results = Path(sys.argv[1])
rows, molecules, names = [], [], []
for path in sorted((results / "generated/8w1l").glob("*_generated_*.sdf")):
    mol = Chem.SDMolSupplier(str(path))[0]
    if mol is None:
        raise ValueError(f"Invalid SDF: {path}")
    rows.append((path.name, Chem.MolToSmiles(mol), round(Descriptors.MolWt(mol), 3),
                 round(QED.qed(mol), 4), "not_measured"))
    molecules.append(mol)
    names.append(path.stem)
with (results / "generated_properties.csv").open("w", newline="") as handle:
    csv.writer(handle).writerows([("sdf", "smiles", "molecular_weight_da", "qed", "assay"), *rows])
Draw.MolsToGridImage(molecules, molsPerRow=3, subImgSize=(300, 230), legends=names).save(
    str(results / "generated_molecules.png"))
''', OUTPUT])
display(pd.read_csv(OUTPUT / 'generated_properties.csv'))
display(Image(filename=str(OUTPUT / 'generated_molecules.png')))


### Vina 계산 점수

공식 평가기가 생성 포즈를 채점하고 국소 최적화합니다. 논문이 사용한 GLIDE와 Sanofi 화합물 검색은 아래 논문 결과에만 포함됩니다. Vina 점수와 실험 IC₅₀는 단위와 의미가 다릅니다.

In [ ]:
stage('vina', [PYTHON, '-m', 'scripts.conDitar.evaluate_mol',
               '--sample_path', SAMPLE_ROOT, '--protein_root', INPUTS,
               '--docking_mode', 'vina_score', '--tmp_dir', RUN / 'vina_tmp'])
run([PYTHON, '-c', '''
import csv
from pathlib import Path
import sys
import torch

results = Path(sys.argv[1])
records = torch.load(results / "generated/eval_results/metrics_-1.pt",
                     map_location="cpu", weights_only=False)["all_results"]
with (results / "vina_scores.csv").open("w", newline="") as handle:
    writer = csv.writer(handle)
    writer.writerow(("sdf", "vina_pose_score_kcal_mol", "vina_minimized_score_kcal_mol"))
    for record in records:
        writer.writerow((record["ligand_filename"], record["vina"]["score_only"][0]["affinity"],
                         record["vina"]["minimize"][0]["affinity"]))
''', OUTPUT], cwd=REPO)
display(pd.read_csv(OUTPUT / 'vina_scores.csv'))


### 논문의 CSF1R 기능 억제 실험값

같은 단백질을 다룬 원 논문의 결과입니다. CL-0은 이번 실행의 포켓을 지정하는 결정구조 리간드입니다. CL-1과 CL-2는 생성 모델의 결과를 단서로 화합물 라이브러리에서 검색했고, CL-10과 CL-11은 모델이 직접 생성했습니다. 이번 실행에서 새로 생성한 분자의 실험값은 미측정으로 저장합니다.

In [ ]:
import matplotlib.pyplot as plt

PAPER = 'https://arxiv.org/html/2607.12349v1#S5'
assays = pd.DataFrame([
    ('CL-0', '8W1L reference ligand', 20, 'measured'),
    ('CL-1', 'shape search hit', 200, 'measured'),
    ('CL-2', 'shape search hit', 309, 'measured'),
    ('CL-10', 'direct model generation', None, 'inactive'),
    ('CL-11', 'direct model generation', None, 'inactive'),
    ('CL-12', 'direct model generation', None, 'not determined'),
], columns=['paper_candidate', 'origin', 'ic50_nm', 'assay_status'])
assays['source'] = PAPER
assays.to_csv(OUTPUT / 'paper_csf1r_table6.csv', index=False)
display(assays)
measured = assays.dropna(subset=['ic50_nm'])
fig, ax = plt.subplots(figsize=(6, 3))
ax.scatter(measured['ic50_nm'], measured['paper_candidate'], s=55)
ax.set(xscale='log', xlabel='Measured CSF1R IC50 (nM)', title='Published CSF1R assay, Table 6')
ax.grid(axis='x', alpha=.3)
fig.tight_layout()
fig.savefig(OUTPUT / 'paper_csf1r_ic50.png', dpi=160)
plt.show()


In [ ]:
from google.colab import files

archive = shutil.make_archive(str(RUN / 'conditar_v001_results'), 'zip', root_dir=OUTPUT)
files.download(archive)
